In [1]:
%reload_ext autoreload
%autoreload 2

from scipy.constants import mu_0
import MaNTA
from Objective import make_objective


import jax
import jax.numpy as jnp
import equinox as eqx
import yancc

from yancc_wrapper import yancc_data

import desc
from desc import set_device
set_device("gpu")

from desc.equilibrium import Equilibrium
from desc.geometry import FourierRZToroidalSurface
from desc.grid import Grid, LinearGrid
from desc.objectives import (
    AspectRatio,
    FixBoundaryR,
    FixBoundaryZ,
    FixCurrent,
    FixPsi,
    ForceBalance,
    LinearObjectiveFromUser,
    ObjectiveFunction,
    ObjectiveFromUser,
    RotationalTransform,
    Volume,
)
from desc.profiles import SplineProfile



Using gpu implementation
Using cache directory: /global/cfs/cdirs/mp217/eatocco/__pycache__
[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]


In [2]:
from Stellarator import StellaratorTransport

st_config = {
    "SourceCenter": 0.0,
    "SourceHeight": 80.0,
    "SourceWidth": 0.2,
    "EdgeTemperature":0.1,
    "EdgeDensity": 0.0,
    "n0": 0.5,
}
# runner = MaNTA.Runner(st)

# # %%
solver_config = {
    "OutputFilename": "stellarator_opt",
    "Polynomial_degree": 3,
    "Grid_size": 6,
    "tau": 100.0, 
    "Lower_boundary": 0.0,
    "Upper_boundary": 0.9,
    "Relative_tolerance": 0.01,
    "delta_t": 0.001,
    "restart": True,
    "solveAdjoint": False, 
}

config = {
    "Stellarator": st_config,
    "Solver": solver_config,
}

Density = lambda x : (st_config["n0"] - st_config["EdgeDensity"]) * (1 - x * x) + st_config["EdgeDensity"]

points =  MaNTA.getNodes(solver_config["Lower_boundary"], solver_config["Upper_boundary"], solver_config["Grid_size"], solver_config["Polynomial_degree"])

yancc_rho = jnp.array(points)
yancc_ntheta = 17
yancc_nzeta = 33

# to allow maximum flexibility to match manta, we use a spline with the same control points as manta \
# + axis and lcfs
# initial pressure is all zeros, can change this if desired
pressure_rho = jnp.concatenate([jnp.zeros(1), yancc_rho, jnp.ones(1)])
desc_pressure = SplineProfile(jnp.zeros_like(pressure_rho), pressure_rho)

eq_est = desc.examples.get("ESTELL")
surf = eq_est.get_surface_at(rho=1)
eq = Equilibrium(M=4, N=4, Psi=0.087, surface=surf, pressure=desc_pressure)
eq = eq.solve(x_scale="ess")[0]

eq_init = eq.copy()

V0 = eq.compute("V")["V"]
# yancc_wrapper = yancc_data.from_eq(points, grid = yancc_grid,rho = yancc_rho, Density=Density, eq=eq_init, nt = yancc_ntheta, nz = yancc_nzeta)
yancc_wrapper = yancc_data.from_eq(points, eq=eq_init, Density=Density)

st = StellaratorTransport(config, yancc_wrapper=yancc_wrapper)
st.run()

st_config = {
    "SourceCenter": 0.0,
    "SourceHeight": 80.0,
    "SourceWidth": 0.2,
    "EdgeTemperature":0.1,
    "EdgeDensity": 0.0,
    "n0": 0.5,
}
# runner = MaNTA.Runner(st)

# # %%
solver_config = {
    "OutputFilename": "stellarator_opt",
    "Polynomial_degree": 3,
    "Grid_size": 6,
    "tau": 100.0, 
    "Lower_boundary": 0.0,
    "Upper_boundary": 0.9,
    "Relative_tolerance": 0.01,
    "delta_t": 0.001,
    "restart": True,
    "solveAdjoint": True, 
}

config = {
    "Stellarator": st_config,
    "Solver": solver_config,
}

Objective = make_objective(config, vectorized=True)



Building objective: force
Precomputing transforms
Building objective: lcfs R
Building objective: lcfs Z
Building objective: fixed Psi
Building objective: fixed pressure
Building objective: fixed current
Building objective: fixed sheet current
Building objective: self_consistency R
Building objective: self_consistency Z
Building objective: lambda gauge
Building objective: axis R self consistency
Building objective: axis Z self consistency
Number of parameters: 120
Number of objectives: 850

Starting optimization
Using method: lsq-exact
Optimization terminated successfully.
`ftol` condition satisfied. (ftol=1.00e-02)
         Current function value: 2.817e-05
         Total delta_x: 4.630e-01
         Iterations: 7
         Function evaluations: 8
         Jacobian evaluations: 8
                                                                 Start  -->   End
Total (sum of squares):                                      2.754e+00  -->   2.817e-05, 
Maximum absolute Force error:          

INFO: Using default value for configuration option RestartFile
Total HDG degrees of freedom 79
INFO: Using default value for configuration option Absolute_tolerance
INFO: Using default value for configuration option tZero
INFO: Using default value for configuration option MinStepSize
INFO: Using default value for configuration option OutputPoints
INFO: Using default value for configuration option SteadyStateTolerance
INFO: Using default value for configuration option WriteOutput
Configuration done.
Setting initial conditions
Residual norm at t = 0: inf


Number of Residual Evaluations due to IDACalcIC 0


Residual norm at t = 0: 0.0525152
Residual norm at t = 0.001: inf
Residual norm at t = 0.001: 0.052516


Writing output at 0.001 ( 1 timesteps )
 dy/dt norm inferred from lambdas is 0.00254385
Steady State achieved at time t = 0.001
Total Number of Timesteps             :1
Total Number of Residual Evaluations  :1
Total Number of Jacobian Computations :1
Done.


In [3]:
# @eqx.filter_custom_jvp
# def Objective(fields, grid, yin):
#     yancc_wrapper = yancc_data.from_other(fields, grid, yin)

#     solver_config = {
#         "OutputFilename": "stellarator_opt",
#         "Polynomial_degree": 3,
#         "Grid_size": 6,
#         "tau": 100.0, 
#         "Lower_boundary": 0.0,
#         "Upper_boundary": 0.9,
#         "Relative_tolerance": 0.01,
#         "delta_t": 0.001,
#         "restart": True,
#         "solveAdjoint": True, 
#     }

#     config = {
#         "Stellarator": st_config,
#         "Solver": solver_config,
#     }
#     st = StellaratorTransport(config, yancc_wrapper=yancc_wrapper)
#     st.runner.Run_ss()
#     G, G_p = st.runAdjointSolve()

#     pi = jnp.array(st.getPressure())

#     return G[0], pi

# @Objective.def_jvp
# def Objective_jvp(primals, tangents):
#     fields, grid, yin = primals
#     field_dot,_,_ = tangents
#     yancc_wrapper = yancc_data.from_other(fields, grid, yin)

#     solver_config = {
#         "OutputFilename": "stellarator_opt",
#         "Polynomial_degree": 3,
#         "Grid_size": 6,
#         "tau": 100.0, 
#         "Lower_boundary": 0.0,
#         "Upper_boundary": 0.9,
#         "Relative_tolerance": 0.01,
#         "delta_t": 0.001,
#         "restart": True,
#         "solveAdjoint": True, 
#     }

#     config = {
#         "Stellarator": st_config,
#         "Solver": solver_config,
#     }
#     st = StellaratorTransport(config, yancc_wrapper=yancc_wrapper)
#     st.runner.Run_ss()
#     G, G_p = st.runAdjointSolve()
#     pi = jnp.array(st.getPressure())
    
#     field_dot_flatten,_ = jax.flatten_util.ravel_pytree(field_dot)

#     return (G[0], pi), (jnp.float32(jnp.dot(G_p.flatten(), field_dot_flatten)), None)

In [4]:
def manta_yancc_fun(fields, grid, yancc_wrapper):

    stored_energy, pressure = Objective(fields, grid, yancc_wrapper) 

    return stored_energy, pressure

@eqx.filter_jit
def objective_from_user_fun(grid, data):
  # note: don't change the signature to this function
    yancc_dat = {
        "B_sup_t": data["B^theta"],
        "B_sup_z": data["B^zeta"],
        "B_sub_t": data["B_theta"],
        "B_sub_z": data["B_zeta"],
        "Bmag": data["|B|"],
        "dBdt": data["|B|_t"],
        "dBdz": data["|B|_z"],
        "sqrtg": data["sqrt(g)"],
    }

    yancc_dat = {
        key: grid.meshgrid_reshape(val, "rtz") for key, val in yancc_dat.items()
    }

    yancc_dat["Psi"] = grid.compress(
        data["Psi"] / grid.nodes[:, 0] ** 2, surface_label="rho"
    )
    yancc_dat["a_minor"] = jnp.full(grid.num_rho, data["a"])
    yancc_dat["R_major"] = jnp.full(grid.num_rho, data["R0"])
    yancc_dat["iota"] = grid.compress(data["iota"], surface_label="rho")
    yancc_dat["rho"] = grid.compress(grid.nodes[:, 0], surface_label="rho")

    yancc_fields = jax.vmap(lambda d: yancc.field.Field(**d, NFP=grid.NFP))(yancc_dat)
    # yancc_fields = desc.backend.tree_unstack(yancc_fields)

    stored_energy, manta_pressure = manta_yancc_fun(yancc_fields, grid, yancc_wrapper)
    
    desc_pressure = grid.compress(data["p"], surface_label="rho")
    pressure_error = manta_pressure - desc_pressure

    # optimization is easiest for least squares objectives, so instead of maximizing
    # stored energy we minimize 1/stored_energy^2 (the squaring happens later)
    return jnp.append(pressure_error, 1 / stored_energy)

    


In [5]:
# pressure_rho = jnp.concatenate([jnp.zeros(1), yancc_rho, jnp.ones(1)])
# desc_pressure = SplineProfile(jnp.zeros_like(pressure_rho), pressure_rho)

# # create initial equilibrium. Psi chosen to give B ~ 1 T.
# # M=N=4 is fine for quick testing, for actual optimization may want to increase to ~8-10
# eq = Equilibrium(M=4, N=4, Psi=0.087, surface=surf, pressure=desc_pressure)
# eq = eq.solve(x_scale="ess")[0]
# # store initial equilibrium for comparison later
# eq_init = eq.copy()
# V0 = eq.compute("V")["V"]
yancc_ntheta = 17
yancc_nzeta = 33
yancc_rho = yancc_wrapper.rho
pressure_rho = jnp.concatenate([jnp.zeros(1), yancc_rho, jnp.ones(1)])
desc_pressure = SplineProfile(jnp.zeros_like(pressure_rho), pressure_rho)

# grid where desc needs to evaluate field for yancc/manta
yancc_desc_grid = LinearGrid(
    rho=yancc_rho, theta=yancc_ntheta, zeta=yancc_nzeta, NFP=eq.NFP
)

def pressure_constraint_fun(params):
    # function to fix dp/dr=0 at axis and p=0 at edge
    # can modify this for other BC (eg fix p at rho=0.8)
    p_l = params["p_l"]
    dp0 = desc_pressure(Grid(jnp.zeros((1, 3)), jitable=True), p_l, dr=1)
    p1 = desc_pressure(Grid(jnp.zeros((1, 3)).at[0, 0].set(1.0), jitable=True), p_l)
    return jnp.array([dp0, p1]).squeeze()


pressure_constraint_target = jnp.array([0.0, 0.0])

# initial optimization just to get self consistent pressure with fixed initial boundary
pressure_error_weight = jnp.full(yancc_desc_grid.num_rho, 1)
stored_energy_weight = 0
objective_from_user_weight = jnp.append(pressure_error_weight, stored_energy_weight)

objectives = [
    ObjectiveFromUser(
        objective_from_user_fun,
        eq,
        target=0,
        weight=objective_from_user_weight,
        grid=yancc_desc_grid,
        deriv_mode="fwd", 
        use_jit=False,# need this assuming manta only has vjp, if using jvp switch to fwd
    )
]

constraints = [
    ForceBalance(eq=eq),  # J x B - grad(p) = 0
    FixCurrent(eq=eq),  # fix zero current, eventually should use real bootstrap
    FixPsi(eq=eq),  # fix total magnetic flux
    FixBoundaryR(eq=eq),  # fix boundary shape
    FixBoundaryZ(eq=eq),
    LinearObjectiveFromUser(
        pressure_constraint_fun, eq, target=pressure_constraint_target
    ),
]

objective = ObjectiveFunction(objectives)

eq, info_out = eq.optimize(
    objective=objective,
    constraints=constraints,
    optimizer="proximal-lsq-exact",
    maxiter=50,
    verbose=3,
    x_scale="ess",
    copy=True,
    options={
        # pressure is O(1e4) so we use a larger trust region for the self consistency part
        "initial_trust_radius": 1e3,
        "max_trust_radius": 1e5,
    },
)
# save for later
eq_self_consistent_pressure = eq.copy()


# main optimization, varying boundary to maximize stored energy


# other objectives are non-dimensionalized, so weights should account for that
# and handle relative weighting, this will likely need trial and error
pressure_error_weight = jnp.full(yancc_desc_grid.num_rho, 10)
stored_energy_weight = 1000
objective_from_user_weight = jnp.append(pressure_error_weight, stored_energy_weight)

objectives = [
    AspectRatio(eq=eq, target=6, weight=10),
    Volume(eq=eq, target=V0, weight=10),
    RotationalTransform(eq=eq, target=0.42, weight=10),
    ObjectiveFromUser(
        objective_from_user_fun,
        eq,
        target=0,
        weight=objective_from_user_weight,
        grid=yancc_desc_grid,
        deriv_mode="fwd", 
        use_jit=False,# need this assuming manta only has vjp, if using jvp switch to fwd
    ),
]
constraints = [
    ForceBalance(eq=eq),  # J x B - grad(p) = 0
    FixCurrent(eq=eq),  # fix zero current, eventually should use real bootstrap
    FixPsi(eq=eq),  # fix total magnetic flux
    LinearObjectiveFromUser(
        pressure_constraint_fun, eq, target=pressure_constraint_target
    ),
]

objective = ObjectiveFunction(objectives)

eq, info_out = eq.optimize(
    objective=objective,
    constraints=constraints,
    optimizer="proximal-lsq-exact",
    maxiter=50,
    verbose=3,
    x_scale="ess",
    copy=True,
)

eq_optimized = eq.copy()


# do a final pass with just the self consistency part to make sure profiles match
pressure_error_weight = jnp.full(yancc_desc_grid.num_rho, 1)
stored_energy_weight = 0
objective_from_user_weight = jnp.append(pressure_error_weight, stored_energy_weight)

objectives = [
    ObjectiveFromUser(
        objective_from_user_fun,
        eq,
        target=0,
        weight=objective_from_user_weight,
        grid=yancc_desc_grid,
        deriv_mode="fwd", 
        use_jit=False,# need this assuming manta only has vjp, if using jvp switch to fwd
    )
]

constraints = [
    ForceBalance(eq=eq),  # J x B - grad(p) = 0
    FixCurrent(eq=eq),  # fix zero current, eventually should use real bootstrap
    FixPsi(eq=eq),  # fix total magnetic flux
    FixBoundaryR(eq=eq),  # fix boundary shape
    FixBoundaryZ(eq=eq),
    LinearObjectiveFromUser(
        pressure_constraint_fun, eq, target=pressure_constraint_target
    ),
]

objective = ObjectiveFunction(objectives)

eq, info_out = eq.optimize(
    objective=objective,
    constraints=constraints,
    optimizer="proximal-lsq-exact",
    maxiter=50,
    verbose=3,
    x_scale="ess",
    copy=True,
    options={
        "initial_trust_radius": 1e3,
        "max_trust_radius": 1e5,
    },
)
eq_optimized_self_consistent = eq.copy()

Building objective: Custom
yancc_wrapper initialized successfully.
---------
False
--------
Timer: Objective build = 11.8 sec
Building objective: force
Precomputing transforms
Timer: Precomputing transforms = 99.1 ms
Timer: Objective build = 113 ms
Timer: Objective build = 1.49 ms
Timer: Eq Update LinearConstraintProjection build = 134 ms
Timer: Proximal projection build = 16.2 sec
Building objective: fixed current
Building objective: fixed Psi
Building objective: lcfs R
Building objective: lcfs Z
Building objective: custom linear
Timer: Objective build = 2.12 sec
Timer: LinearConstraintProjection build = 5.97 sec
Number of parameters: 24
Number of objectives: 25
Timer: Initializing the optimization = 25.0 sec

Starting optimization
Using method: proximal-lsq-exact
yancc_wrapper initialized successfully.
configuring
Successfully created StellaratorTransport object


INFO: Using default value for configuration option RestartFile
Total HDG degrees of freedom 79
INFO: Using default value for configuration option Absolute_tolerance
INFO: Using default value for configuration option tZero
INFO: Using default value for configuration option MinStepSize
INFO: Using default value for configuration option OutputPoints
INFO: Using default value for configuration option SteadyStateTolerance
INFO: Using default value for configuration option WriteOutput
Configuration done.
Setting initial conditions
ERROR:2026-04-24 05:15:53,524:jax._src.callback:442: jax.io_callback failed
Traceback (most recent call last):
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/jax/_src/callback.py", line 440, in io_callback_impl
    return tree_util.tree_map(np.asarray, callback(*args))
                                          ^^^^^^^^^^^^^^^
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/jax/_src/callback.py", l

JaxRuntimeError: INTERNAL: CpuCallback error calling callback: Traceback (most recent call last):
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/runpy.py", line 198, in _run_module_as_main
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/runpy.py", line 88, in _run_code
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
  File "/global/homes/e/eatocco/.conda/en

In [ ]:
eq_optimized_self_consistent.save("optimized_equilibrium.h5")